# DCIT 411: Bioinformatics Project
# Sequence Alignment with Biopython

**Student Name:** [Your Name]

**Date:** February 27, 2026

---

## Project Overview
This project explores various sequence alignment techniques using Biopython, including:
- Pairwise sequence alignment (Needleman-Wunsch, Smith-Waterman)
- Multiple sequence alignment (Progressive MSA)
- Advanced topics (Consensus sequences, conservation analysis)

## Table of Contents
1. [Literature Review and Theoretical Background](#1-literature-review)
2. [Data Collection and Preprocessing](#2-data-collection)
3. [Pairwise Sequence Alignment](#3-pairwise-alignment)
4. [Multiple Sequence Alignment](#4-multiple-alignment)
5. [Advanced Topics](#5-advanced-topics)
6. [Results and Discussion](#6-results)
7. [Conclusions](#7-conclusions)

---
## 1. Literature Review and Theoretical Background

### 1.1 Sequence Alignment Fundamentals

**Sequence alignment** is the process of arranging sequences of DNA, RNA, or protein to identify regions of similarity. These similarities may indicate functional, structural, or evolutionary relationships between sequences.

### 1.2 Dynamic Programming Algorithms

#### Needleman-Wunsch Algorithm (Global Alignment)
- **Purpose:** Aligns sequences from end to end
- **Use case:** Comparing sequences of similar length and similar overall composition
- **Method:** Uses dynamic programming to find optimal alignment with maximum score
- **Time Complexity:** O(mn) where m and n are sequence lengths

#### Smith-Waterman Algorithm (Local Alignment)
- **Purpose:** Finds the best matching subsequence region between two sequences
- **Use case:** Finding conserved domains or motifs
- **Method:** Similar to Needleman-Wunsch but allows negative scores to reset to zero
- **Applications:** Database searching, detecting domains, finding conserved regions

### 1.3 Substitution Matrices

**Substitution matrices** provide scores for aligning amino acid or nucleotide pairs:

- **BLOSUM (BLOcks SUbstitution Matrix):**
  - BLOSUM62: Most commonly used, based on sequences with ≤62% identity
  - Higher numbers (BLOSUM80): For closely related sequences
  - Lower numbers (BLOSUM45): For distantly related sequences

- **PAM (Point Accepted Mutation):**
  - PAM1: 1% amino acid change
  - Higher numbers indicate more evolutionary distance

### 1.4 Gap Penalties

Gaps represent insertions or deletions (indels) in evolution:
- **Gap open penalty:** Cost to start a gap (typically -10 to -15)
- **Gap extension penalty:** Cost to extend an existing gap (typically -0.5 to -2)
- **Affine gap penalty:** More biologically realistic (penalizes opening gaps more than extending)

### 1.5 Biological Significance

Sequence alignment enables:
1. **Homology detection:** Identifying genes with common ancestry
2. **Functional annotation:** Predicting protein function from similar sequences
3. **Structural prediction:** Similar sequences often have similar structures
4. **Evolutionary studies:** Understanding phylogenetic relationships
5. **Conserved motif detection:** Finding functionally important regions

---
## 2. Data Collection and Preprocessing

### 2.1 Setup and Installation

In [ ]:
# Install required packages (uncomment if running on Colab)
# !pip install biopython matplotlib pandas numpy

# For Colab: Install MSA tools (optional - we'll use Python-only implementation)
# !apt-get update
# !apt-get install -y clustalw muscle mafft

In [ ]:
# Import required libraries
from Bio import Entrez, SeqIO, Align, AlignIO, Phylo
from Bio.Align import substitution_matrices, MultipleSeqAlignment, PairwiseAligner
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from collections import Counter
import time
import warnings
warnings.filterwarnings('ignore')

print("✓ All libraries imported successfully")

### 2.2 Data Collection from NCBI

We will fetch hemoglobin sequences from NCBI's protein database.

In [ ]:
def fetch_sequences_from_ncbi(ids, email, output_file="sequences.fasta"):
    """
    Fetch protein sequences from NCBI using Entrez
    """
    Entrez.email = email
    
    sequences = []
    print(f"Fetching {len(ids)} sequences from NCBI...")
    
    for acc_id in ids:
        try:
            handle = Entrez.efetch(db="protein", id=acc_id, rettype="fasta", retmode="text")
            record = SeqIO.read(handle, "fasta")
            sequences.append(record)
            handle.close()
            print(f"  ✓ Fetched {acc_id}: {record.description}")
            time.sleep(0.34)  # NCBI rate limiting
        except Exception as e:
            print(f"  ✗ Failed to fetch {acc_id}: {e}")
    
    # Save to file
    SeqIO.write(sequences, output_file, "fasta")
    print(f"\n✓ Saved {len(sequences)} sequences to '{output_file}'")
    
    return sequences

# Example IDs (Hemoglobin sequences from different species)
sequence_ids = [
    "NP_000509.1",  # Human hemoglobin subunit beta
    "NP_000518.1",  # Human LDL receptor (for comparison)
    "NP_032244.2",  # Mouse hemoglobin subunit alpha
    "NP_001107728.1",  # Xenopus aquaporin-8
    "NP_001094389.1"   # Sheep thyroid hormone receptor alpha
]

# Fetch sequences
sequences = fetch_sequences_from_ncbi(sequence_ids, "your.email@example.com", "raw_sequences.fasta")

### 2.3 Sequence Preprocessing

Clean and prepare sequences for alignment by:
- Removing gaps and special characters
- Filtering by length
- Handling ambiguous residues

In [ ]:
def preprocess_sequences(input_file, output_file, min_length=50, max_length=1000):
    """
    Preprocess sequences: remove gaps, filter by length
    """
    sequences = list(SeqIO.parse(input_file, "fasta"))
    processed = []
    
    print(f"Processing {len(sequences)} sequences...")
    print(f"Filters: {min_length} ≤ length ≤ {max_length}")
    
    for record in sequences:
        # Remove gaps and special characters
        clean_seq = str(record.seq).replace("-", "").replace("*", "").upper()
        
        # Filter by length
        if min_length <= len(clean_seq) <= max_length:
            # Create new record with cleaned sequence
            clean_record = SeqRecord(
                Seq(clean_seq),
                id=record.id,
                description=record.description
            )
            processed.append(clean_record)
            print(f"  ✓ {record.id}: {len(clean_seq)} residues")
        else:
            print(f"  ✗ {record.id}: {len(clean_seq)} residues (filtered out)")
    
    # Save processed sequences
    SeqIO.write(processed, output_file, "fasta")
    print(f"\n✓ Saved {len(processed)} processed sequences to '{output_file}'")
    
    return processed

# Preprocess sequences
processed_sequences = preprocess_sequences("raw_sequences.fasta", "hemoglobin_processed.fasta")

### 2.4 Exploratory Data Analysis

In [ ]:
# Analyze sequence properties
seq_data = []
for record in processed_sequences:
    seq_str = str(record.seq)
    seq_data.append({
        'ID': record.id,
        'Description': record.description[:50] + '...',
        'Length': len(seq_str),
        'GC Content': round((seq_str.count('G') + seq_str.count('C')) / len(seq_str) * 100, 2)
    })

df = pd.DataFrame(seq_data)
print("\nSequence Statistics:")
print(df.to_string(index=False))

# Visualize sequence lengths
plt.figure(figsize=(10, 4))
plt.bar(df['ID'], df['Length'], color='steelblue')
plt.xlabel('Sequence ID')
plt.ylabel('Length (residues)')
plt.title('Sequence Length Distribution')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---
## 3. Pairwise Sequence Alignment

### 3.1 Needleman-Wunsch (Global Alignment)

Global alignment aligns sequences from end to end, finding the best overall alignment.

In [ ]:
class PairwiseAlignerAnalyzer:
    def __init__(self):
        self.aligner = PairwiseAligner()
        self.results = []
    
    def load_sequences(self, fasta_file, seq1_id, seq2_id):
        """Load two sequences for pairwise alignment"""
        sequences = SeqIO.to_dict(SeqIO.parse(fasta_file, "fasta"))
        self.seq1 = sequences[seq1_id]
        self.seq2 = sequences[seq2_id]
        return self
    
    def needleman_wunsch_global(self, matrix_name="BLOSUM62", open_gap=-10, extend_gap=-0.5):
        """Perform global alignment using Needleman-Wunsch"""
        self.aligner.mode = "global"
        self.aligner.open_gap_score = open_gap
        self.aligner.extend_gap_score = extend_gap
        self.aligner.substitution_matrix = substitution_matrices.load(matrix_name)
        
        start_time = time.time()
        alignments = self.aligner.align(self.seq1.seq, self.seq2.seq)
        best_alignment = alignments[0]
        end_time = time.time()
        
        # Calculate statistics
        aligned_seq1, aligned_seq2 = str(best_alignment[0]), str(best_alignment[1])
        matches = sum(1 for a, b in zip(aligned_seq1, aligned_seq2) if a == b and a != '-')
        length = len(aligned_seq1)
        gaps = aligned_seq1.count('-') + aligned_seq2.count('-')
        
        result = {
            'method': 'Needleman-Wunsch (Global)',
            'matrix': matrix_name,
            'score': best_alignment.score,
            'identity': matches / length * 100,
            'gaps': gaps,
            'length': length,
            'time': end_time - start_time,
            'alignment': best_alignment
        }
        self.results.append(result)
        
        print(f"\n=== Global Alignment (Needleman-Wunsch) ===")
        print(f"Matrix: {matrix_name}, Gap penalties: open={open_gap}, extend={extend_gap}")
        print(f"Score: {result['score']:.1f}")
        print(f"Identity: {result['identity']:.2f}%")
        print(f"Gaps: {result['gaps']}")
        print(f"Alignment length: {result['length']}")
        print(f"Time: {result['time']:.4f} seconds")
        
        return result
    
    def smith_waterman_local(self, matrix_name="BLOSUM62", open_gap=-10, extend_gap=-0.5):
        """Perform local alignment using Smith-Waterman"""
        self.aligner.mode = "local"
        self.aligner.open_gap_score = open_gap
        self.aligner.extend_gap_score = extend_gap
        self.aligner.substitution_matrix = substitution_matrices.load(matrix_name)
        
        start_time = time.time()
        alignments = self.aligner.align(self.seq1.seq, self.seq2.seq)
        best_alignment = alignments[0]
        end_time = time.time()
        
        # Calculate statistics
        aligned_seq1, aligned_seq2 = str(best_alignment[0]), str(best_alignment[1])
        matches = sum(1 for a, b in zip(aligned_seq1, aligned_seq2) if a == b and a != '-')
        length = len(aligned_seq1)
        gaps = aligned_seq1.count('-') + aligned_seq2.count('-')
        
        result = {
            'method': 'Smith-Waterman (Local)',
            'matrix': matrix_name,
            'score': best_alignment.score,
            'identity': matches / length * 100,
            'gaps': gaps,
            'length': length,
            'time': end_time - start_time,
            'alignment': best_alignment
        }
        self.results.append(result)
        
        print(f"\n=== Local Alignment (Smith-Waterman) ===")
        print(f"Matrix: {matrix_name}, Gap penalties: open={open_gap}, extend={extend_gap}")
        print(f"Score: {result['score']:.1f}")
        print(f"Identity: {result['identity']:.2f}%")
        print(f"Gaps: {result['gaps']}")
        print(f"Alignment length: {result['length']}")
        print(f"Time: {result['time']:.4f} seconds")
        
        return result
    
    def print_alignment(self, result, width=60):
        """Print alignment in a readable format"""
        alignment = result['alignment']
        aligned_seq1, aligned_seq2 = str(alignment[0]), str(alignment[1])
        
        print(f"\n{result['method']} - First {width} positions:")
        print(f"Seq1: {aligned_seq1[:width]}")
        print(f"      {''.join(['|' if a==b else ' ' for a,b in zip(aligned_seq1[:width], aligned_seq2[:width])])}")
        print(f"Seq2: {aligned_seq2[:width]}")

# Perform pairwise alignments
analyzer = PairwiseAlignerAnalyzer()
analyzer.load_sequences("hemoglobin_processed.fasta", "NP_000509.1", "NP_032244.2")

# Global alignment
global_result = analyzer.needleman_wunsch_global()
analyzer.print_alignment(global_result)

# Local alignment
local_result = analyzer.smith_waterman_local()
analyzer.print_alignment(local_result)

### 3.2 Experimenting with Different Parameters

In [ ]:
# Test different substitution matrices
print("\n=== Testing Different Substitution Matrices ===")
matrices = ["BLOSUM62", "BLOSUM80", "BLOSUM45", "PAM250"]

matrix_results = []
for matrix in matrices:
    try:
        result = analyzer.needleman_wunsch_global(matrix_name=matrix)
        matrix_results.append({
            'Matrix': matrix,
            'Score': f"{result['score']:.1f}",
            'Identity %': f"{result['identity']:.2f}",
            'Time (s)': f"{result['time']:.4f}"
        })
    except:
        print(f"Matrix {matrix} not available")

df_matrices = pd.DataFrame(matrix_results)
print("\nComparison of Substitution Matrices:")
print(df_matrices.to_string(index=False))

In [ ]:
# Test different gap penalties
print("\n=== Testing Different Gap Penalties ===")
gap_penalties = [
    (-10, -0.5),
    (-5, -0.5),
    (-15, -1.0),
    (-8, -2.0)
]

gap_results = []
for open_gap, extend_gap in gap_penalties:
    result = analyzer.needleman_wunsch_global(open_gap=open_gap, extend_gap=extend_gap)
    gap_results.append({
        'Open': open_gap,
        'Extend': extend_gap,
        'Score': f"{result['score']:.1f}",
        'Identity %': f"{result['identity']:.2f}",
        'Gaps': result['gaps']
    })

df_gaps = pd.DataFrame(gap_results)
print("\nComparison of Gap Penalties:")
print(df_gaps.to_string(index=False))

### 3.3 Visualizing Alignment Results

In [ ]:
# Create comparison visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Compare global vs local
methods = ['Global\n(Needleman-Wunsch)', 'Local\n(Smith-Waterman)']
scores = [global_result['score'], local_result['score']]
identities = [global_result['identity'], local_result['identity']]

axes[0].bar(methods, scores, color=['steelblue', 'coral'])
axes[0].set_ylabel('Alignment Score')
axes[0].set_title('Alignment Scores')
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(methods, identities, color=['steelblue', 'coral'])
axes[1].set_ylabel('Identity (%)')
axes[1].set_title('Sequence Identity')
axes[1].set_ylim(0, 100)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 4. Multiple Sequence Alignment (MSA)

### 4.1 Progressive MSA Implementation

Progressive alignment builds a multiple sequence alignment by:
1. Computing pairwise distances
2. Building a guide tree (UPGMA)
3. Progressively aligning sequences following the tree

In [ ]:
class ProgressiveMSA:
    """Progressive Multiple Sequence Alignment using Biopython"""
    
    def __init__(self, sequences, matrix_name="BLOSUM62"):
        self.sequences = sequences
        self.aligner = PairwiseAligner()
        self.aligner.mode = 'global'
        self.aligner.substitution_matrix = substitution_matrices.load(matrix_name)
        self.aligner.open_gap_score = -10
        self.aligner.extend_gap_score = -0.5
        print(f"Initialized Progressive MSA with {len(sequences)} sequences")
    
    def calculate_pairwise_distances(self):
        """Calculate distance matrix between all sequence pairs"""
        n = len(self.sequences)
        distances = np.zeros((n, n))
        
        print("\nCalculating pairwise distances...")
        for i in range(n):
            for j in range(i+1, n):
                alignments = self.aligner.align(self.sequences[i].seq, self.sequences[j].seq)
                score = alignments[0].score
                
                # Convert score to distance
                max_len = max(len(self.sequences[i].seq), len(self.sequences[j].seq))
                distance = 1 - (score / (max_len * 5))
                
                distances[i, j] = distance
                distances[j, i] = distance
        
        return distances
    
    def guide_tree_upgma(self, distances):
        """Build UPGMA guide tree"""
        n = len(self.sequences)
        clusters = [[i] for i in range(n)]
        
        while len(clusters) > 1:
            min_dist = float('inf')
            merge_i, merge_j = 0, 1
            
            for i in range(len(clusters)):
                for j in range(i+1, len(clusters)):
                    dist_sum = sum(distances[seq_i, seq_j] 
                                 for seq_i in clusters[i] 
                                 for seq_j in clusters[j])
                    avg_dist = dist_sum / (len(clusters[i]) * len(clusters[j]))
                    
                    if avg_dist < min_dist:
                        min_dist = avg_dist
                        merge_i, merge_j = i, j
            
            new_cluster = clusters[merge_i] + clusters[merge_j]
            clusters = [c for idx, c in enumerate(clusters) if idx not in [merge_i, merge_j]]
            clusters.append(new_cluster)
        
        return clusters[0]
    
    def get_consensus(self, alignment):
        """Get consensus sequence from alignment"""
        consensus = []
        aln_length = len(str(alignment[0].seq))
        
        for pos in range(aln_length):
            column = [str(rec.seq)[pos] for rec in alignment if str(rec.seq)[pos] != '-']
            if column:
                counts = Counter(column)
                consensus.append(counts.most_common(1)[0][0])
            else:
                consensus.append('-')
        
        return Seq(''.join(consensus))
    
    def progressive_alignment(self):
        """Perform progressive multiple sequence alignment"""
        print("\n" + "="*50)
        print("Progressive Multiple Sequence Alignment")
        print("="*50)
        
        start_time = time.time()
        
        # Step 1: Pairwise distances
        distances = self.calculate_pairwise_distances()
        
        # Step 2: Guide tree
        print("Building guide tree (UPGMA)...")
        order = self.guide_tree_upgma(distances)
        
        # Step 3: Progressive alignment
        print("Performing progressive alignment...")
        
        # Start with first pair
        first_pair = self.aligner.align(self.sequences[order[0]].seq, 
                                        self.sequences[order[1]].seq)[0]
        
        alignment = [
            SeqRecord(Seq(str(first_pair[0])), id=self.sequences[order[0]].id, 
                     description=self.sequences[order[0]].description),
            SeqRecord(Seq(str(first_pair[1])), id=self.sequences[order[1]].id,
                     description=self.sequences[order[1]].description)
        ]
        
        # Add remaining sequences
        for idx in order[2:]:
            print(f"  Adding sequence {idx+1}/{len(self.sequences)}: {self.sequences[idx].id}")
            new_seq = self.sequences[idx]
            consensus = self.get_consensus(alignment)
            pair_aln = self.aligner.align(consensus, new_seq.seq)[0]
            
            aligned_consensus = str(pair_aln[0])
            aligned_new = str(pair_aln[1])
            
            # Add gaps to existing alignment
            gap_positions = [i for i, c in enumerate(aligned_consensus) 
                           if c == '-' and aligned_new[i] != '-']
            
            new_alignment = []
            for record in alignment:
                seq_str = str(record.seq)
                for gap_pos in sorted(gap_positions):
                    seq_str = seq_str[:gap_pos] + '-' + seq_str[gap_pos:]
                new_alignment.append(SeqRecord(Seq(seq_str), id=record.id, 
                                             description=record.description))
            
            new_alignment.append(SeqRecord(Seq(aligned_new), id=new_seq.id, 
                                         description=new_seq.description))
            alignment = new_alignment
        
        end_time = time.time()
        
        msa = MultipleSeqAlignment(alignment)
        
        print(f"\n✓ Alignment completed in {end_time - start_time:.2f} seconds")
        print(f"  Alignment length: {msa.get_alignment_length()}")
        print(f"  Number of sequences: {len(msa)}")
        
        return msa, end_time - start_time

# Load sequences and perform MSA
sequences = list(SeqIO.parse("hemoglobin_processed.fasta", "fasta"))
progressive_msa = ProgressiveMSA(sequences)
msa_alignment, comp_time = progressive_msa.progressive_alignment()

### 4.2 MSA Analysis and Visualization

In [ ]:
def calculate_conservation(alignment):
    """Calculate conservation score for each position"""
    conservation = []
    aln_length = alignment.get_alignment_length()
    
    for i in range(aln_length):
        column = alignment[:, i]
        column_no_gaps = [c for c in column if c != '-']
        
        if len(column_no_gaps) == 0:
            conservation.append(0)
        else:
            counts = Counter(column_no_gaps)
            most_common = counts.most_common(1)[0][1]
            conservation.append(most_common / len(column_no_gaps))
    
    return conservation

# Calculate conservation
conservation = calculate_conservation(msa_alignment)

print(f"\nConservation Statistics:")
print(f"  Mean conservation: {np.mean(conservation):.3f}")
print(f"  Max conservation: {np.max(conservation):.3f}")
print(f"  Min conservation: {np.min(conservation):.3f}")
print(f"  Std conservation: {np.std(conservation):.3f}")

In [ ]:
# Print alignment excerpt
print("\n" + "="*70)
print("ALIGNMENT EXCERPT (First 60 positions)")
print("="*70)

for record in msa_alignment:
    seq_str = str(record.seq)[:60]
    print(f"{record.id:20s} {seq_str}")

In [ ]:
# Comprehensive MSA visualization
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# 1. Conservation profile
axes[0].plot(conservation, color='blue', linewidth=1.5)
axes[0].fill_between(range(len(conservation)), conservation, alpha=0.3)
axes[0].set_xlabel('Alignment Position')
axes[0].set_ylabel('Conservation Score')
axes[0].set_title('Sequence Conservation Profile')
axes[0].set_ylim(0, 1)
axes[0].grid(True, alpha=0.3)

# 2. Alignment heatmap
aa_to_num = {aa: i+1 for i, aa in enumerate('ACDEFGHIKLMNPQRSTVWY')}
aa_to_num['-'] = 0

aln_array = []
for record in msa_alignment:
    seq_nums = [aa_to_num.get(aa, 0) for aa in str(record.seq)]
    aln_array.append(seq_nums)

im = axes[1].imshow(aln_array, aspect='auto', cmap='viridis', interpolation='nearest')
axes[1].set_xlabel('Alignment Position')
axes[1].set_ylabel('Sequence')
axes[1].set_title('Alignment Heatmap')
axes[1].set_yticks(range(len(msa_alignment)))
axes[1].set_yticklabels([rec.id for rec in msa_alignment], fontsize=8)
plt.colorbar(im, ax=axes[1], label='Amino Acid')

# 3. Gap distribution
gap_counts = [str(rec.seq).count('-') for rec in msa_alignment]
seq_ids = [rec.id for rec in msa_alignment]

axes[2].barh(seq_ids, gap_counts, color='coral')
axes[2].set_xlabel('Number of Gaps')
axes[2].set_ylabel('Sequence')
axes[2].set_title('Gap Distribution per Sequence')
axes[2].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('msa_visualization.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Visualization saved as 'msa_visualization.png'")

### 4.3 Comparing with Professional MSA Tools (ClustalW, MUSCLE, MAFFT)

When running on platforms where external tools are available (like Google Colab or Linux systems), we can compare our Python implementation with industry-standard MSA tools.

In [ ]:
import subprocess
import os
import shutil

def check_tool_available(tool_name):
    """Check if an MSA tool is installed"""
    return shutil.which(tool_name) is not None

class ProfessionalMSAComparison:
    """Run and compare professional MSA tools"""
    
    def __init__(self, input_file):
        self.input_file = input_file
        self.results = {}
        
    def run_clustalw(self):
        """Run ClustalW alignment"""
        if not check_tool_available("clustalw2"):
            print("⊗ ClustalW not available (skip or install with: apt install clustalw)")
            return None
            
        print("\n" + "="*50)
        print("Running ClustalW (Progressive Method)")
        print("="*50)
        
        output_file = "clustalw_output.aln"
        
        try:
            cmd = [
                "clustalw2",
                f"-INFILE={self.input_file}",
                f"-OUTFILE={output_file}",
                "-OUTPUT=CLUSTAL",
                "-OUTORDER=INPUT",
                "-QUIET"
            ]
            
            start_time = time.time()
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
            end_time = time.time()
            
            if result.returncode == 0 and os.path.exists(output_file):
                alignment = AlignIO.read(output_file, "clustal")
                conservation = calculate_conservation(alignment)
                
                self.results['ClustalW'] = {
                    'alignment': alignment,
                    'time': end_time - start_time,
                    'conservation': conservation,
                    'mean_conservation': np.mean(conservation)
                }
                
                print(f"✓ ClustalW completed in {end_time - start_time:.2f} seconds")
                print(f"  Alignment length: {alignment.get_alignment_length()}")
                print(f"  Mean conservation: {np.mean(conservation):.3f}")
                return alignment
            else:
                print(f"✗ ClustalW failed: {result.stderr}")
                return None
                
        except subprocess.TimeoutExpired:
            print("✗ ClustalW timed out")
            return None
        except Exception as e:
            print(f"✗ Error running ClustalW: {e}")
            return None
    
    def run_muscle(self):
        """Run MUSCLE alignment"""
        if not check_tool_available("muscle"):
            print("⊗ MUSCLE not available (skip or install with: apt install muscle)")
            return None
            
        print("\n" + "="*50)
        print("Running MUSCLE (Iterative Method)")
        print("="*50)
        
        output_file = "muscle_output.aln"
        
        try:
            cmd = [
                "muscle",
                "-in", self.input_file,
                "-out", output_file,
                "-maxiters", "16",
                "-quiet"
            ]
            
            start_time = time.time()
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
            end_time = time.time()
            
            if result.returncode == 0 and os.path.exists(output_file):
                alignment = AlignIO.read(output_file, "fasta")
                conservation = calculate_conservation(alignment)
                
                self.results['MUSCLE'] = {
                    'alignment': alignment,
                    'time': end_time - start_time,
                    'conservation': conservation,
                    'mean_conservation': np.mean(conservation)
                }
                
                print(f"✓ MUSCLE completed in {end_time - start_time:.2f} seconds")
                print(f"  Alignment length: {alignment.get_alignment_length()}")
                print(f"  Mean conservation: {np.mean(conservation):.3f}")
                return alignment
            else:
                print(f"✗ MUSCLE failed: {result.stderr}")
                return None
                
        except subprocess.TimeoutExpired:
            print("✗ MUSCLE timed out")
            return None
        except Exception as e:
            print(f"✗ Error running MUSCLE: {e}")
            return None
    
    def run_mafft(self):
        """Run MAFFT alignment"""
        if not check_tool_available("mafft"):
            print("⊗ MAFFT not available (skip or install with: apt install mafft)")
            return None
            
        print("\n" + "="*50)
        print("Running MAFFT (Fast Fourier Transform)")
        print("="*50)
        
        output_file = "mafft_output.aln"
        
        try:
            cmd = [
                "mafft",
                "--auto",
                "--quiet",
                self.input_file
            ]
            
            start_time = time.time()
            with open(output_file, 'w') as f:
                result = subprocess.run(cmd, stdout=f, stderr=subprocess.PIPE, text=True, timeout=120)
            end_time = time.time()
            
            if result.returncode == 0 and os.path.exists(output_file):
                alignment = AlignIO.read(output_file, "fasta")
                conservation = calculate_conservation(alignment)
                
                self.results['MAFFT'] = {
                    'alignment': alignment,
                    'time': end_time - start_time,
                    'conservation': conservation,
                    'mean_conservation': np.mean(conservation)
                }
                
                print(f"✓ MAFFT completed in {end_time - start_time:.2f} seconds")
                print(f"  Alignment length: {alignment.get_alignment_length()}")
                print(f"  Mean conservation: {np.mean(conservation):.3f}")
                return alignment
            else:
                print(f"✗ MAFFT failed: {result.stderr.decode()}")
                return None
                
        except subprocess.TimeoutExpired:
            print("✗ MAFFT timed out")
            return None
        except Exception as e:
            print(f"✗ Error running MAFFT: {e}")
            return None
    
    def compare_all_methods(self):
        """Run all available methods and compare"""
        print("\n" + "="*70)
        print("COMPARING MSA METHODS")
        print("="*70)
        
        # Add Python implementation result
        self.results['Python (Custom)'] = {
            'alignment': msa_alignment,
            'time': comp_time,
            'conservation': conservation,
            'mean_conservation': np.mean(conservation)
        }
        
        # Run professional tools
        self.run_clustalw()
        self.run_muscle()
        self.run_mafft()
        
        if len(self.results) > 1:
            # Create comparison table
            comparison_data = []
            for method, data in self.results.items():
                comparison_data.append({
                    'Method': method,
                    'Time (s)': f"{data['time']:.3f}",
                    'Alignment Length': data['alignment'].get_alignment_length(),
                    'Mean Conservation': f"{data['mean_conservation']:.3f}"
                })
            
            df_comparison = pd.DataFrame(comparison_data)
            print("\n" + "="*70)
            print("METHOD COMPARISON TABLE")
            print("="*70)
            print(df_comparison.to_string(index=False))
            
            # Visualize comparison
            self.visualize_comparison()
            
            return df_comparison
        else:
            print("\n⚠ No professional tools available for comparison")
            print("Note: This is expected on Windows without WSL")
            print("On Google Colab, the tools are installed and comparison will be shown")
            return None
    
    def visualize_comparison(self):
        """Create comparison visualizations"""
        if len(self.results) < 2:
            return
        
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        
        methods = list(self.results.keys())
        colors = ['purple', 'blue', 'green', 'orange'][:len(methods)]
        
        # 1. Time comparison
        times = [self.results[m]['time'] for m in methods]
        axes[0, 0].bar(range(len(methods)), times, color=colors, alpha=0.7)
        axes[0, 0].set_xticks(range(len(methods)))
        axes[0, 0].set_xticklabels(methods, rotation=45, ha='right')
        axes[0, 0].set_ylabel('Time (seconds)')
        axes[0, 0].set_title('Execution Time Comparison')
        axes[0, 0].grid(axis='y', alpha=0.3)
        
        # 2. Conservation profiles
        for i, method in enumerate(methods):
            cons = self.results[method]['conservation']
            axes[0, 1].plot(cons, label=method, alpha=0.7, color=colors[i], linewidth=1.5)
        axes[0, 1].set_xlabel('Position')
        axes[0, 1].set_ylabel('Conservation')
        axes[0, 1].set_title('Conservation Profiles Comparison')
        axes[0, 1].legend()
        axes[0, 1].set_ylim(0, 1)
        axes[0, 1].grid(alpha=0.3)
        
        # 3. Alignment length comparison
        lengths = [self.results[m]['alignment'].get_alignment_length() for m in methods]
        axes[1, 0].bar(range(len(methods)), lengths, color=colors, alpha=0.7)
        axes[1, 0].set_xticks(range(len(methods)))
        axes[1, 0].set_xticklabels(methods, rotation=45, ha='right')
        axes[1, 0].set_ylabel('Alignment Length')
        axes[1, 0].set_title('Alignment Length Comparison')
        axes[1, 0].grid(axis='y', alpha=0.3)
        
        # 4. Mean conservation comparison
        mean_cons = [self.results[m]['mean_conservation'] for m in methods]
        axes[1, 1].bar(range(len(methods)), mean_cons, color=colors, alpha=0.7)
        axes[1, 1].set_xticks(range(len(methods)))
        axes[1, 1].set_xticklabels(methods, rotation=45, ha='right')
        axes[1, 1].set_ylabel('Mean Conservation')
        axes[1, 1].set_title('Mean Conservation Comparison')
        axes[1, 1].set_ylim(0, 1)
        axes[1, 1].grid(axis='y', alpha=0.3)
        
        plt.tight_layout()
        plt.savefig('msa_methods_comparison.png', dpi=300, bbox_inches='tight')
        plt.show()
        
        print("\n✓ Comparison visualization saved as 'msa_methods_comparison.png'")

# Run comparison
msa_comparison = ProfessionalMSAComparison("hemoglobin_processed.fasta")
comparison_results = msa_comparison.compare_all_methods()

### 4.4 Pairwise Identity Matrix

In [ ]:
def calculate_pairwise_identity_matrix(alignment):
    """Calculate pairwise identity between all sequences"""
    n = len(alignment)
    identity_matrix = np.zeros((n, n))
    
    for i in range(n):
        for j in range(i, n):
            if i == j:
                identity_matrix[i, j] = 100
            else:
                seq_i = str(alignment[i].seq)
                seq_j = str(alignment[j].seq)
                
                matches = sum(1 for a, b in zip(seq_i, seq_j) if a == b and a != '-')
                length = sum(1 for a, b in zip(seq_i, seq_j) if a != '-' or b != '-')
                
                identity = (matches / length * 100) if length > 0 else 0
                identity_matrix[i, j] = identity
                identity_matrix[j, i] = identity
    
    return identity_matrix

# Calculate and visualize identity matrix
identity_matrix = calculate_pairwise_identity_matrix(msa_alignment)
seq_ids = [rec.id for rec in msa_alignment]

plt.figure(figsize=(10, 8))
im = plt.imshow(identity_matrix, cmap='YlOrRd', vmin=0, vmax=100)
plt.colorbar(im, label='Identity (%)')
plt.xticks(range(len(seq_ids)), seq_ids, rotation=45, ha='right')
plt.yticks(range(len(seq_ids)), seq_ids)
plt.title('Pairwise Sequence Identity Matrix')

# Add text annotations
for i in range(len(seq_ids)):
    for j in range(len(seq_ids)):
        text = plt.text(j, i, f'{identity_matrix[i, j]:.1f}',
                       ha="center", va="center", color="black", fontsize=8)

plt.tight_layout()
plt.savefig('identity_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Identity matrix saved as 'identity_matrix.png'")

---
## 5. Advanced Topics

### 5.1 Consensus Sequence Generation

In [ ]:
def generate_consensus_sequence(alignment, threshold=0.5):
    """Generate consensus sequence from MSA"""
    consensus = []
    consensus_info = []
    aln_length = alignment.get_alignment_length()
    
    for pos in range(aln_length):
        column = alignment[:, pos]
        column_no_gaps = [c for c in column if c != '-']
        
        if len(column_no_gaps) == 0:
            consensus.append('-')
            consensus_info.append({'position': pos+1, 'residue': '-', 'frequency': 0, 'conservation': 0})
        else:
            counts = Counter(column_no_gaps)
            most_common_residue, most_common_count = counts.most_common(1)[0]
            frequency = most_common_count / len(column_no_gaps)
            
            if frequency >= threshold:
                consensus.append(most_common_residue)
            else:
                consensus.append('X')  # Ambiguous position
            
            consensus_info.append({
                'position': pos+1,
                'residue': most_common_residue,
                'frequency': frequency,
                'conservation': frequency
            })
    
    consensus_seq = ''.join(consensus)
    return consensus_seq, consensus_info

# Generate consensus
consensus_seq, consensus_info = generate_consensus_sequence(msa_alignment)

print("\n" + "="*70)
print("CONSENSUS SEQUENCE")
print("="*70)
print(f"\nConsensus (first 100 residues):")
print(consensus_seq[:100])
print(f"\nFull length: {len(consensus_seq)} residues")
print(f"Ambiguous positions (X): {consensus_seq.count('X')}")
print(f"Gap positions (-): {consensus_seq.count('-')}")

# Show highly conserved positions
highly_conserved = [info for info in consensus_info if info['conservation'] > 0.8 and info['residue'] != '-']
print(f"\nHighly conserved positions (>80%): {len(highly_conserved)}")
print("\nTop 10 most conserved positions:")
for info in sorted(highly_conserved, key=lambda x: x['conservation'], reverse=True)[:10]:
    print(f"  Position {info['position']}: {info['residue']} ({info['conservation']*100:.1f}%)")

### 5.2 Identifying Conserved Motifs

In [ ]:
def find_conserved_motifs(alignment, window_size=5, threshold=0.7):
    """Find conserved motifs/regions in the alignment"""
    conservation = calculate_conservation(alignment)
    motifs = []
    
    for i in range(len(conservation) - window_size + 1):
        window_conservation = conservation[i:i+window_size]
        avg_conservation = np.mean(window_conservation)
        
        if avg_conservation >= threshold:
            # Extract motif from first sequence
            motif_seq = str(msa_alignment[0].seq)[i:i+window_size]
            motifs.append({
                'position': i+1,
                'end_position': i+window_size,
                'motif': motif_seq,
                'conservation': avg_conservation
            })
    
    # Merge overlapping motifs
    merged_motifs = []
    if motifs:
        current_motif = motifs[0]
        for motif in motifs[1:]:
            if motif['position'] <= current_motif['end_position']:
                # Extend current motif
                current_motif['end_position'] = max(current_motif['end_position'], motif['end_position'])
                current_motif['conservation'] = max(current_motif['conservation'], motif['conservation'])
            else:
                merged_motifs.append(current_motif)
                current_motif = motif
        merged_motifs.append(current_motif)
    
    return merged_motifs

# Find conserved motifs
conserved_motifs = find_conserved_motifs(msa_alignment)

print("\n" + "="*70)
print("CONSERVED MOTIFS")
print("="*70)
print(f"\nFound {len(conserved_motifs)} conserved regions (≥70% conservation)\n")

for i, motif in enumerate(conserved_motifs[:10], 1):
    print(f"Motif {i}:")
    print(f"  Position: {motif['position']}-{motif['end_position']}")
    print(f"  Length: {motif['end_position'] - motif['position'] + 1}")
    print(f"  Conservation: {motif['conservation']*100:.1f}%")
    print()

### 5.3 Phylogenetic Analysis

In [ ]:
# Create simple phylogenetic tree from distance matrix
def create_simple_tree(alignment):
    """Create a simple distance-based phylogenetic tree"""
    from Bio import Phylo
    from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor
    
    # Calculate distance matrix
    calculator = DistanceCalculator('identity')
    dm = calculator.get_distance(alignment)
    
    # Construct tree using UPGMA
    constructor = DistanceTreeConstructor()
    tree = constructor.upgma(dm)
    
    return tree

try:
    # Build phylogenetic tree
    tree = create_simple_tree(msa_alignment)
    
    print("\n" + "="*70)
    print("PHYLOGENETIC TREE (UPGMA)")
    print("="*70)
    
    # Draw tree
    fig, ax = plt.subplots(figsize=(10, 6))
    Phylo.draw(tree, axes=ax, do_show=False)
    plt.title('Phylogenetic Tree of Sequences')
    plt.tight_layout()
    plt.savefig('phylogenetic_tree.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✓ Tree saved as 'phylogenetic_tree.png'")
    
    # Print tree in text format
    print("\nTree structure:")
    Phylo.draw_ascii(tree)
    
except Exception as e:
    print(f"Could not create phylogenetic tree: {e}")

### 5.4 Profile-Based Analysis

In [ ]:
def create_position_frequency_matrix(alignment):
    """Create position frequency matrix (profile)"""
    amino_acids = list('ACDEFGHIKLMNPQRSTVWY')
    aln_length = alignment.get_alignment_length()
    
    profile = {aa: [0] * aln_length for aa in amino_acids}
    profile['-'] = [0] * aln_length
    
    for pos in range(aln_length):
        column = alignment[:, pos]
        counts = Counter(column)
        total = len(column)
        
        for aa, count in counts.items():
            if aa in profile:
                profile[aa][pos] = count / total
    
    return profile

# Create profile
profile = create_position_frequency_matrix(msa_alignment)

print("\n" + "="*70)
print("POSITION FREQUENCY MATRIX (PROFILE)")
print("="*70)
print(f"\nCreated profile for {msa_alignment.get_alignment_length()} positions")

# Visualize profile for first 50 positions
amino_acids = list('ACDEFGHIKLMNPQRSTVWY')
profile_array = np.array([profile[aa][:50] for aa in amino_acids])

plt.figure(figsize=(14, 8))
plt.imshow(profile_array, aspect='auto', cmap='Blues', interpolation='nearest')
plt.colorbar(label='Frequency')
plt.yticks(range(len(amino_acids)), amino_acids)
plt.xlabel('Position (first 50)')
plt.ylabel('Amino Acid')
plt.title('Position Frequency Matrix (Profile) - First 50 Positions')
plt.tight_layout()
plt.savefig('profile_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Profile visualization saved as 'profile_matrix.png'")

---
## 6. Results and Discussion

### 6.1 Summary of Findings

In [ ]:
print("="*70)
print("COMPREHENSIVE RESULTS SUMMARY")
print("="*70)

print("\n### 1. Pairwise Alignment Results")
print(f"\nGlobal Alignment (Needleman-Wunsch):")
print(f"  Score: {global_result['score']:.1f}")
print(f"  Identity: {global_result['identity']:.2f}%")
print(f"  Gaps: {global_result['gaps']}")

print(f"\nLocal Alignment (Smith-Waterman):")
print(f"  Score: {local_result['score']:.1f}")
print(f"  Identity: {local_result['identity']:.2f}%")
print(f"  Gaps: {local_result['gaps']}")

print("\n### 2. Multiple Sequence Alignment Results")
print(f"  Number of sequences: {len(msa_alignment)}")
print(f"  Alignment length: {msa_alignment.get_alignment_length()}")
print(f"  Mean conservation: {np.mean(conservation):.3f}")
print(f"  Computation time: {comp_time:.2f} seconds")

print("\n### 3. Conservation Analysis")
print(f"  Highly conserved regions: {len([c for c in conservation if c > 0.8])}")
print(f"  Poorly conserved regions: {len([c for c in conservation if c < 0.3])}")
print(f"  Conserved motifs identified: {len(conserved_motifs)}")

print("\n### 4. Sequence Diversity")
mean_identity = np.mean([identity_matrix[i,j] for i in range(len(identity_matrix)) 
                        for j in range(i+1, len(identity_matrix))])
print(f"  Mean pairwise identity: {mean_identity:.2f}%")
print(f"  Max pairwise identity: {np.max(identity_matrix[np.triu_indices_from(identity_matrix, k=1)]):.2f}%")
print(f"  Min pairwise identity: {np.min(identity_matrix[np.triu_indices_from(identity_matrix, k=1)]):.2f}%")

### 6.2 Discussion

#### Pairwise Alignment Insights
- **Global vs Local:** Global alignment considers the entire sequence length, making it suitable for sequences of similar length and composition. Local alignment focuses on finding the best matching region, which is more appropriate for identifying conserved domains.
- **Substitution Matrices:** BLOSUM62 is most commonly used and works well for moderately diverged sequences. Different matrices can reveal different evolutionary relationships.
- **Gap Penalties:** Appropriate gap penalties are crucial. Too lenient penalties lead to excessive gaps; too stringent penalties may miss important insertions/deletions.

#### Multiple Sequence Alignment Quality
- **Conservation Patterns:** High conservation scores indicate functionally or structurally important regions that are under selective pressure.
- **Gap Distribution:** Gaps often correspond to indels in evolution or flexible loop regions in protein structures.
- **Computational Efficiency:** Progressive alignment is relatively fast but may be suboptimal compared to iterative methods like MUSCLE.

#### Biological Significance
- **Conserved Motifs:** Highly conserved regions often represent:
  - Active sites in enzymes
  - Binding sites for ligands
  - Structural elements crucial for protein folding
- **Sequence Diversity:** Low pairwise identity suggests:
  - Distant evolutionary relationships
  - Functional divergence
  - Different structural adaptations

### 6.3 Challenges and Limitations

1. **Progressive Alignment Limitations:**
   - Errors in early alignments propagate through the process
   - May not find globally optimal alignment
   - Sensitive to guide tree accuracy

2. **Computational Complexity:**
   - Dynamic programming has O(n²) time complexity
   - MSA becomes computationally expensive with many sequences
   - Memory requirements increase quadratically

3. **Parameter Selection:**
   - Choice of substitution matrix affects results
   - Gap penalty selection is often empirical
   - No single "correct" parameter set for all cases

4. **Alignment Ambiguity:**
   - Multiple alignments may have similar scores
   - Difficult to assess alignment confidence
   - Particularly challenging in poorly conserved regions

### 6.4 Potential Improvements

1. **Iterative Refinement:**
   - Implement iterative methods (like MUSCLE) to improve alignment quality
   - Use consistency-based scoring (like T-Coffee)

2. **Profile-based Methods:**
   - Implement PSI-BLAST for remote homology detection
   - Use Hidden Markov Models (HMMer) for better profile representation

3. **Structural Information:**
   - Incorporate 3D structure data when available
   - Use structure-based alignment tools (DALI, TM-align)

4. **Statistical Validation:**
   - Implement bootstrapping to assess alignment confidence
   - Calculate statistical significance of alignment scores
   - Use cross-validation to evaluate parameter choices

---
## 7. Conclusions

This project successfully implemented and analyzed various sequence alignment techniques using Biopython:

### Key Achievements

1. **Pairwise Alignment:**
   - Implemented both global (Needleman-Wunsch) and local (Smith-Waterman) alignment algorithms
   - Evaluated impact of different substitution matrices and gap penalties
   - Demonstrated the trade-offs between global and local alignment approaches

2. **Multiple Sequence Alignment:**
   - Developed a progressive alignment implementation using Python/Biopython
   - Successfully aligned multiple protein sequences
   - Identified conserved regions and motifs

3. **Advanced Analysis:**
   - Generated consensus sequences
   - Calculated conservation scores across alignment positions
   - Created position frequency matrices (profiles)
   - Built phylogenetic trees to visualize evolutionary relationships

4. **Practical Applications:**
   - Demonstrated how to fetch sequences from NCBI
   - Showed preprocessing steps for real biological data
   - Created comprehensive visualizations for result interpretation

### Biological Insights

- Sequence alignment reveals evolutionary relationships and functional similarities
- Conserved regions often indicate functionally important residues
- Different alignment methods are appropriate for different biological questions
- Gap patterns can reveal important evolutionary events (insertions/deletions)

### Technical Skills Developed

- Proficiency with Biopython for bioinformatics tasks
- Understanding of dynamic programming algorithms
- Data visualization using matplotlib
- Interpretation of alignment quality metrics
- Working with biological databases (NCBI)

### Future Directions

Future work could extend this project by:
- Implementing more sophisticated MSA algorithms (MAFFT, Clustal Omega)
- Adding structural alignment capabilities
- Incorporating machine learning for alignment quality prediction
- Developing a web interface for interactive analysis
- Applying these methods to larger-scale comparative genomics studies

---

### References

1. Needleman, S. B., & Wunsch, C. D. (1970). A general method applicable to the search for similarities in the amino acid sequence of two proteins. *Journal of Molecular Biology*, 48(3), 443-453.

2. Smith, T. F., & Waterman, M. S. (1981). Identification of common molecular subsequences. *Journal of Molecular Biology*, 147(1), 195-197.

3. Henikoff, S., & Henikoff, J. G. (1992). Amino acid substitution matrices from protein blocks. *Proceedings of the National Academy of Sciences*, 89(22), 10915-10919.

4. Thompson, J. D., Higgins, D. G., & Gibson, T. J. (1994). CLUSTAL W: improving the sensitivity of progressive multiple sequence alignment through sequence weighting, position-specific gap penalties and weight matrix choice. *Nucleic Acids Research*, 22(22), 4673-4680.

5. Edgar, R. C. (2004). MUSCLE: multiple sequence alignment with high accuracy and high throughput. *Nucleic Acids Research*, 32(5), 1792-1797.

6. Cock, P. J., Antao, T., Chang, J. T., Chapman, B. A., Cox, C. J., Dalke, A., ... & de Hoon, M. J. (2009). Biopython: freely available Python tools for computational molecular biology and bioinformatics. *Bioinformatics*, 25(11), 1422-1423.

---

**Project completed on February 27, 2026**

---

## Appendix: Export Alignment Results

In [ ]:
# Export MSA to different formats
AlignIO.write(msa_alignment, "msa_output.aln", "clustal")
AlignIO.write(msa_alignment, "msa_output.fasta", "fasta")

print("✓ Alignment exported to:")
print("  - msa_output.aln (Clustal format)")
print("  - msa_output.fasta (FASTA format)")

# Save consensus sequence
with open("consensus_sequence.txt", "w") as f:
    f.write(">Consensus_Sequence\n")
    f.write(consensus_seq)

print("  - consensus_sequence.txt")

# Save statistics to CSV
stats_data = []
for i, record in enumerate(msa_alignment):
    seq_str = str(record.seq)
    stats_data.append({
        'Sequence_ID': record.id,
        'Length': len(seq_str),
        'Gaps': seq_str.count('-'),
        'Gap_Percent': f"{seq_str.count('-')/len(seq_str)*100:.2f}"
    })

df_stats = pd.DataFrame(stats_data)
df_stats.to_csv("alignment_statistics.csv", index=False)
print("  - alignment_statistics.csv")

print("\n✓ All results exported successfully!")